# Notebook 04: ArcGIS Web Map and Dashboard

**Obstacle-Aware Clustering for Geographic Data**

This notebook turns the clustering results from Notebook 03 into an interactive ArcGIS Online deliverable. Python handles the data preparation and publishing; the Map Viewer and Dashboards designer in AGOL handle the styling and layout.

### What gets built

1. A hosted feature layer with all 1,068 clustered fires, including the basin-wide cluster labels, the near-shore (2 km) flag, and popup-friendly attribute columns.
2. Two CSV tables -- per-cluster summaries and view-level metrics -- that the dashboard's side panel and headline indicators pull from.
3. A web map styled in the AGOL Map Viewer with a unique-value renderer in the project's cluster colors and a configured popup.
4. A dashboard assembled in the AGOL Dashboards designer with a side panel that updates on cluster click and a toggle between the basin-wide and near-shore views.

## 1. Setup

In [1]:
# Standard packages
import getpass
from pathlib import Path

import numpy as np
import pandas as pd

# ArcGIS Python API -- only what we need for content management
from arcgis.gis import GIS

# Cluster colors from the project palette
# (We don't use these in code -- they're here as a reference for the Map Viewer step)
CLUSTER_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
print('Cluster colors for the Map Viewer:')
for i, c in enumerate(CLUSTER_COLORS, start=1):
    print(f'  Cluster {i}: {c}')

Cluster colors for the Map Viewer:
  Cluster 1: #e74c3c
  Cluster 2: #3498db
  Cluster 3: #2ecc71
  Cluster 4: #f39c12


## 2. Loading Results from Notebook 03

Notebook 03 saved three files to `data/processed/`:

- `tahoe_fires_clustered.csv`: one row per fire, with all clustering labels (basin-wide and near-shore), the near-shore flag, and the original attributes.
- `cluster_summary.csv`: one row per cluster per view (4 basin + 4 near-shore = 8 rows). Used for the dashboard's side panel.
- `global_metrics.csv`: one row per view (basin and near-shore) with optimal weights, $\sigma_a$, and the $s$ contribution percentages. Used for the dashboard's headline indicators.

In [2]:
processed_dir = Path('../data/processed')

fires = pd.read_csv(processed_dir / 'tahoe_fires_clustered.csv')
cluster_summary = pd.read_csv(processed_dir / 'cluster_summary.csv')
global_metrics = pd.read_csv(processed_dir / 'global_metrics.csv')

print(f'Loaded {len(fires)} fires')
print(f'Cluster summary rows: {len(cluster_summary)} (4 basin + 4 near-shore)')
print(f'Global metrics rows: {len(global_metrics)} (basin and near-shore views)')
print()
print('Cluster columns available in fires:')
for col in fires.columns:
    if 'cluster' in col or 'near_shore' in col:
        print(f'  {col}')

Loaded 1068 fires
Cluster summary rows: 8 (4 basin + 4 near-shore)
Global metrics rows: 2 (basin and near-shore views)

Cluster columns available in fires:
  cluster_t2_opt
  cluster_t1_oa
  cluster_t2_std
  cluster_t1_std
  cluster_t2_no_s
  near_shore_2km
  cluster_near_opt


In [3]:
# Headline numbers for the dashboard indicators
global_metrics.round(3)

,view,n_fires,optimal_beta,optimal_gamma,mean_arc_span,sigma_a,s_contribution_span_pct,s_contribution_sigma_pct
0,basin,1068,0.468,0.389,0.351,0.583,2.004,16.667
1,near_shore,468,0.886,0.593,0.339,0.417,26.384,66.667


In [4]:
# Per-cluster numbers for the dashboard side panel
cluster_summary.round(2)

,view,cluster_id,cluster_label,n,centroid_lon,centroid_lat,mean_fire_size,median_fire_size,pct_natural,pct_human,cluster_span
0,basin,0,1,485,-119.99,38.91,8.22,0.1,12.16,87.84,0.25
1,basin,1,2,119,-119.93,39.15,52.42,0.1,62.18,37.82,0.43
2,basin,2,3,233,-120.15,39.02,2.14,0.1,27.04,72.96,0.32
3,basin,3,4,231,-120.07,39.23,1.60,0.1,3.46,96.54,0.41
4,near_shore,0,1,57,-119.95,39.15,108.67,0.1,59.65,40.35,0.53
5,near_shore,1,2,104,-120.13,39.03,3.48,0.1,11.54,88.46,0.31
6,near_shore,2,3,168,-120.06,39.23,1.80,0.1,0.00,100.00,0.26
7,near_shore,3,4,139,-119.98,38.96,0.49,0.1,0.00,100.00,0.25


## 3. Preparing Data for ArcGIS

A few transformations before publishing:

1. Convert 0-indexed cluster IDs to 1-indexed labels for display ("Cluster 1, 2, 3, 4" reads better in popups than "Cluster 0, 1, 2, 3").
2. Add a string label for the near-shore cluster that says "Not in near-shore subset" for fires outside the 2 km zone, so the popup is always meaningful.
3. Add an integer 0/1 flag for the near-shore subset so the dashboard can filter the map with a simple SQL expression.
4. Decode `cause_binary` to "Natural" / "Human-Caused" and rename the specific-cause "Natural" to "Lightning" for clarity.
5. Round fire size to 2 decimals and cast year to a string (otherwise ArcGIS displays "2,003" instead of "2003").
6. Pick the columns that should actually appear in the published layer.

In [5]:
# Work on a copy so we don't mutate the loaded DataFrame
fires_pub = fires.copy()

# 1-indexed cluster labels for display
fires_pub['Cluster_Basin'] = (fires_pub['cluster_t2_opt'] + 1).astype(int)

# Near-shore cluster: convert nullable Int64 to a string label
# (NaN for fires outside the near-shore subset becomes a friendly message)
near_label = fires_pub['cluster_near_opt'] + 1   # nullable Int64 with NaN
fires_pub['Cluster_NearShore_Label'] = (
    near_label.astype('Int64').astype('string').fillna('Not in near-shore subset')
)

# 0/1 flag for filtering in the dashboard
fires_pub['Near_Shore_2km'] = fires_pub['near_shore_2km'].astype(int)

# Cause labels
fires_pub['Cause'] = fires_pub['cause_binary'].map({0: 'Natural', 1: 'Human-Caused'})
fires_pub['Specific_Cause'] = fires_pub['NWCG_GENERAL_CAUSE'].replace('Natural', 'Lightning')

# Round fire size for popup readability
fires_pub['Fire_Size_Acres'] = fires_pub['FIRE_SIZE'].round(2)

# Cast year to string so ArcGIS doesn't render it as "2,003"
fires_pub['Fire_Year'] = fires_pub['FIRE_YEAR'].astype(int).astype(str)

# Columns to publish (LATITUDE/LONGITUDE drive the geocoding)
publish_cols = [
    'LATITUDE', 'LONGITUDE',
    'Cluster_Basin',
    'Cluster_NearShore_Label',
    'Near_Shore_2km',
    'Cause', 'Specific_Cause',
    'Fire_Size_Acres', 'Fire_Year',
]
fires_pub = fires_pub[publish_cols].copy()

# Save to a publishable CSV
publish_path = processed_dir / 'tahoe_fires_webmap.csv'
fires_pub.to_csv(publish_path, index=False)

print(f'Saved {len(fires_pub)} fires to {publish_path.name}')
print()
print('Column dtypes:')
print(fires_pub.dtypes)
print()
print('Basin cluster distribution:')
print(fires_pub['Cluster_Basin'].value_counts().sort_index().to_string())
print()
print(f'Near-shore subset: {fires_pub["Near_Shore_2km"].sum()} fires '
      f'({100 * fires_pub["Near_Shore_2km"].mean():.1f}%)')

Saved 1068 fires to tahoe_fires_webmap.csv

Column dtypes:
LATITUDE                          float64
LONGITUDE                         float64
Cluster_Basin                       int64
Cluster_NearShore_Label    string[python]
Near_Shore_2km                      int64
Cause                              object
Specific_Cause                     object
Fire_Size_Acres                   float64
Fire_Year                          object
dtype: object

Basin cluster distribution:
Cluster_Basin
1    485
2    119
3    233
4    231

Near-shore subset: 468 fires (43.8%)


## 4. Connecting to ArcGIS Online

We authenticate with ArcGIS Online using your credentials. The `getpass` module prompts for your password without displaying it in the notebook output.

**Security note**: Your password is never stored in the notebook — `getpass` handles it securely in memory only.

In [6]:
# Authenticate with ArcGIS Online
gis = GIS(
    url='https://michele-75.maps.arcgis.com',
    username='Michele-75',
    password=getpass.getpass('Enter your ArcGIS Online password: '),
)

print(f'Connected as: {gis.users.me.username}')
print(f'Organization: {gis.properties.name}')

Connected as: Michele-75
Organization: Michele Perry


## 5. Publishing the Hosted Items

We publish three items: the fires feature layer, the cluster summary table, and the global metrics table. The Map Viewer and Dashboards designer pick these up by title in the next sections.

### Re-run safety

If you re-run this notebook, ArcGIS will create duplicate items by default. The cell below checks AGOL for any existing items with our titles and prints what it finds, but does not delete anything. Cleanup is up to you in the AGOL UI.

If you'd rather have the notebook delete duplicates automatically, set `delete_existing = True` in the cell below.

In [7]:
delete_existing = False

# Titles we'll use for the items we're about to create
item_titles = [
    'Tahoe Wildfires - Obstacle-Aware Clustering',   # CSV + feature layer
    'Tahoe Wildfires - Cluster Summary',             # supporting table
    'Tahoe Wildfires - Global Metrics',              # supporting table
]

if delete_existing:
    me = gis.users.me.username
    for title in item_titles:
        # Restrict to our own items so we don't accidentally hit something shared
        existing = gis.content.search(
            query=f'title:"{title}" owner:{me}',
            max_items=20,
        )
        # Search is fuzzy -- filter to exact title match
        existing = [it for it in existing if it.title == title]
        for it in existing:
            print(f'Deleting old item: {it.title} ({it.type}, ID {it.id})')
            it.delete()
    print('Cleanup complete.\n')
else:
    print('Skipping cleanup. New items will be created alongside any existing duplicates.\n')

Skipping cleanup. New items will be created alongside any existing duplicates.



### 5.1 Publish the fires feature layer

In [9]:
# Item properties for the CSV (the resulting feature layer inherits these)
fire_item_properties = {
    'title': 'Tahoe Wildfires - Obstacle-Aware Clustering',
    'description': (
        'Wildfire occurrence data (1992-2020) within the TRPA basin around Lake Tahoe, '
        'clustered with an obstacle-aware k-Means algorithm that respects the lake '
        'as a geographic barrier. Includes basin-wide cluster assignments and a '
        'near-shore (within 2 km of the lake) subset.<br><br>'
        '<b>Source:</b> FPA FOD 6th Edition (USFS Research Data Archive).<br>'
        '<b>Method:</b> Obstacle-aware k-Means with arc-length parameter s along the '
        'Lake Tahoe shoreline. See project repository for full methodology.'
    ),
    'tags': 'wildfires, clustering, Lake Tahoe, obstacle-aware, k-means, TRPA, portfolio',
    'type': 'CSV',
}

print('Uploading fire data to ArcGIS Online...')
fire_csv_item = gis.content.add(fire_item_properties, data=str(publish_path))
print(f'CSV uploaded: {fire_csv_item.title} (ID: {fire_csv_item.id})')

# Publish as a hosted feature layer.
# ArcGIS detects LATITUDE/LONGITUDE and geocodes the points automatically.
print('\nPublishing as hosted feature layer...')
fire_layer_item = fire_csv_item.publish()
print(f'Published: {fire_layer_item.title}')
print(f'Item URL: {fire_layer_item.homepage}')

Uploading fire data to ArcGIS Online...


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


CSV uploaded: Tahoe Wildfires - Obstacle-Aware Clustering (ID: e24da0a4f97744ec923e3962429056fe)

Publishing as hosted feature layer...
Published: Tahoe Wildfires - Obstacle-Aware Clustering
Item URL: https://Michele-75.maps.arcgis.com/home/item.html?id=610df2a34f7f4491a154d23cb2a41024


### 5.2 Publish the supporting tables

In [10]:
# Cluster summary table
summary_item_props = {
    'title': 'Tahoe Wildfires - Cluster Summary',
    'description': (
        'Per-cluster summary statistics for the obstacle-aware clustering of Lake Tahoe '
        'wildfires. Includes both the basin-wide view (4 clusters, 1068 fires) and the '
        'near-shore view (within 2 km of the lake, 4 clusters, 296 fires). '
        'Used to populate the dashboard side panel.'
    ),
    'tags': 'wildfires, clustering, Lake Tahoe, summary, dashboard',
    'type': 'CSV',
}
summary_csv_path = processed_dir / 'cluster_summary.csv'
summary_csv_item = gis.content.add(summary_item_props, data=str(summary_csv_path))
summary_table_item = summary_csv_item.publish()
print(f'Cluster summary published: {summary_table_item.title}')
print(f'  {summary_table_item.homepage}')

# Global metrics table
metrics_item_props = {
    'title': 'Tahoe Wildfires - Global Metrics',
    'description': (
        'View-level metrics for the obstacle-aware clustering of Lake Tahoe wildfires. '
        'Two rows: basin (1068 fires) and near_shore (296 fires within 2 km of the lake). '
        'Includes optimal weights, sigma_a, mean arc-length span, and the s contribution '
        'percentages. Used to populate the dashboard headline indicators.'
    ),
    'tags': 'wildfires, clustering, Lake Tahoe, metrics, dashboard',
    'type': 'CSV',
}
metrics_csv_path = processed_dir / 'global_metrics.csv'
metrics_csv_item = gis.content.add(metrics_item_props, data=str(metrics_csv_path))
metrics_table_item = metrics_csv_item.publish()
print(f'Global metrics published: {metrics_table_item.title}')
print(f'  {metrics_table_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Cluster summary published: Tahoe Wildfires - Cluster Summary
  https://Michele-75.maps.arcgis.com/home/item.html?id=0229330041ff4816b7509c8089f50190


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Global metrics published: Tahoe Wildfires - Global Metrics
  https://Michele-75.maps.arcgis.com/home/item.html?id=71ad25192fd64da8879da4f03dabc543


### 5.3 Share publicly

In [11]:
# Share all three items publicly so anyone with the link can view them
# without an ArcGIS account
for item in [fire_layer_item, summary_table_item, metrics_table_item]:
    item.share(everyone=True)
    print(f'Shared: {item.title}')

print()
print('Item URLs:')
print(f'  Feature layer:   {fire_layer_item.homepage}')
print(f'  Cluster summary: {summary_table_item.homepage}')
print(f'  Global metrics:  {metrics_table_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Tahoe Wildfires - Obstacle-Aware Clustering


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Tahoe Wildfires - Cluster Summary


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Tahoe Wildfires - Global Metrics

Item URLs:
  Feature layer:   https://Michele-75.maps.arcgis.com/home/item.html?id=610df2a34f7f4491a154d23cb2a41024
  Cluster summary: https://Michele-75.maps.arcgis.com/home/item.html?id=0229330041ff4816b7509c8089f50190
  Global metrics:  https://Michele-75.maps.arcgis.com/home/item.html?id=71ad25192fd64da8879da4f03dabc543


## 6. Building the Web Map in the Map Viewer

With the feature layer published, the next step is to style it and save it as a web map. Both happen in the AGOL Map Viewer.

### Step 1: Open the layer in the Map Viewer

1. Go to the feature layer's item page (printed above as "Feature layer").
2. Click **Open in Map Viewer** in the top right.
3. The layer loads with default symbology (small blue dots).

### Step 2: Style the clusters with a unique-value renderer

1. With the fires layer selected in the **Layers** panel, click **Styles** in the right rail.
2. Under **Choose attributes**, click **+ Field** and pick `Cluster_Basin`. The Map Viewer auto-detects this as a category field.
3. Under **Pick a style**, choose **Types (unique symbols)** and click **Style options**.
4. Click each cluster's symbol to set its color from the project palette:
   - Cluster 1 -> `#e74c3c` (red)
   - Cluster 2 -> `#3498db` (blue)
   - Cluster 3 -> `#2ecc71` (green)
   - Cluster 4 -> `#f39c12` (orange)
5. While in the symbol editor for each cluster, set:
   - **Size**: 8 px
   - **Outline color**: dark grey (`#282828` or similar)
   - **Outline width**: 0.5 px
   - **Transparency**: ~14% (this matches the alpha=220 in the static figures)
6. Click **Done** on each panel until the Styles pane closes.

### Step 3: Configure the popup

1. With the fires layer selected, click **Pop-ups** in the right rail and toggle popups on.
2. Set the **Title** to `Fire in Cluster {Cluster_Basin}`.
3. Click **Fields list** and reorder/hide fields so the visible ones are:
   - Cluster_Basin (label: "Basin Cluster")
   - Cluster_NearShore_Label (label: "Near-Shore Cluster")
   - Cause
   - Specific_Cause (label: "Specific Cause")
   - Fire_Size_Acres (label: "Fire Size (acres)", format to 2 decimal places)
   - Fire_Year (label: "Year")
4. Hide LATITUDE, LONGITUDE, Near_Shore_2km, and any auto-added ObjectID fields.

### Step 4: Set the basemap and extent

1. Click **Basemap** in the left rail. Pick **Topographic** (terrain context works well for fire data).
2. Pan/zoom so the TRPA basin fills the view (rough center ~ 39.07 N, -120.03 W; the basin extends roughly 38.85 to 39.30 N).
3. Once the extent looks right, click **Save and open** -> **Save as** in the top toolbar.

### Step 5: Save the web map

1. **Title**: `Tahoe Wildfires - Obstacle-Aware Clustering Web Map`
2. **Tags**: `wildfires, clustering, Lake Tahoe, obstacle-aware, portfolio`
3. **Summary**: `Interactive map of 1,068 Lake Tahoe wildfires (1992-2020) clustered using an obstacle-aware k-Means algorithm.`
4. Click **Save**.

### Step 6: Share publicly

1. From the web map's item page, click **Share** -> set sharing to **Everyone (public)**.
2. Copy the web map URL into the cell below.

In [12]:
# Paste your web map URL here once you've finished building it in the Map Viewer
web_map_url = 'https://michele-75.maps.arcgis.com/home/item.html?id=bd530a6056a8485c89804faf4b782d05'
print(f'Web map URL: {web_map_url}')

Web map URL: https://michele-75.maps.arcgis.com/home/item.html?id=bd530a6056a8485c89804faf4b782d05


## 7. Building the Dashboard in ArcGIS Dashboards

The dashboard is the interactive deliverable. With the web map and supporting tables in place, the layout work below takes about ten minutes.

### Target layout

```
+-----------------------------------------------------+
|  Tahoe Wildfires -- Obstacle-Aware Clustering       |
+-----------------+-----------------------------------+
|  s contributes  |                                   |
|  to span:       |                                   |
|                 |                                   |
|   Basin: 2%     |          Main Map                 |
|   Near-shore:   |       (clustered fires)           |
|        23%      |                                   |
|                 |                                   |
+-----------------+                                   |
|  Cluster List   |                                   |
|  (filtered by   |                                   |
|   selected      |                                   |
|   view)         |                                   |
+-----------------+-----------------------------------+
|        [ Basin ]  [ Near-shore (2 km) ]             |
+-----------------------------------------------------+
```

### Step 1: Create the dashboard

1. Open `https://michele-75.maps.arcgis.com/apps/dashboards/` and click **New dashboard**.
2. Title it **Tahoe Wildfires - Obstacle-Aware Clustering** and click **Create dashboard**.

### Step 2: Add the main map

1. In the dashboard editor, click **+** (top left) -> **Map**.
2. Pick **Tahoe Wildfires - Obstacle-Aware Clustering Web Map**.
3. On the configuration panel, enable **Pop-ups** and **Legend**. Leave the other settings at their defaults.
4. Click **Done**.

### Step 3: Add the two headline indicators

The story to tell is that on the full basin, the arc-length parameter $s$ contributes only ~2% to span improvement, but on the near-shore subset it jumps to ~23% -- an order-of-magnitude difference. Two indicators side by side make this readable at a glance.

For the basin indicator:

1. Click **+** -> **Indicator**.
2. Data source: pick **Tahoe Wildfires - Global Metrics**.
3. **Filter** tab: add a filter where `view is basin`.
4. **Indicator** tab: set **Value** to `s_contribution_span_pct`, **Statistic** to `Sum` (only one row matches the filter, so sum equals the value). Set **Decimal places** to 1.
5. **Top text**: `Basin-wide`.
6. **Middle text** suffix: `%`.
7. **Bottom text**: `s contribution to span`.
8. Click **Done**.

Repeat for the near-shore indicator: same data source, filter `view is near_shore`, top text `Near-shore (2 km)`.

### Step 4: Add the cluster summary list (side panel)

1. Click **+** -> **List**.
2. Data source: **Tahoe Wildfires - Cluster Summary**.
3. **Filter** tab: add a filter where `view is basin` (this is the default; the toggle in Step 5 will flip it).
4. **List** tab -> **Line item template**, paste:
   ```
   <b>Cluster {cluster_label}</b><br>
   n = {n} fires &nbsp;|&nbsp; {pct_human}% human-caused<br>
   Mean fire size: {mean_fire_size} acres
   ```
   (Dashboards supports basic HTML in line items. Click the formatting button on each numeric field to round `pct_human` to integers and `mean_fire_size` to 2 decimals.)
5. **Sort by** `cluster_id` ascending.
6. Click **Done**.

### Step 5: Wire up the basin / near-shore toggle

This is the trickiest step in AGOL Dashboards because we want one selector to do two different things: filter the cluster list by `view`, and filter the map's fire layer by `Near_Shore_2km`. The cleanest approach is a **Category Selector** that drives both targets through configured **actions**.

1. Click **+** -> **Category Selector**.
2. **Categories** tab: select **Defined values** and add two:
   - Label `Basin (all 1,068 fires)`, value `basin`.
   - Label `Near-shore (within 2 km, 296 fires)`, value `near_shore`.
3. **Selector** tab: set **Selection type** to **Single**, **Default selection** to **First**.
4. **Actions** tab -> **Add target** -> pick **Cluster Summary List** -> action **Filter** -> field `view`. The selector value flows straight through (`basin` -> list shows basin rows; `near_shore` -> list shows near-shore rows).
5. **Add target** again -> pick the **Map's fire layer** -> action **Filter** -> field `Near_Shore_2km`. Map the selector values explicitly: `basin` -> `0,1` (both, i.e., no real filter), `near_shore` -> `1`.
6. If your dashboard version doesn't support multi-value mapping in step 5, the simplest workaround is a separate small toggle (a **Number Selector** above the map with a single "Show only near-shore" checkbox) that filters the map independently of the list.
7. Click **Done**.

### Step 6: Save and share

1. Click **Save** in the top right.
2. Click **Share** -> set sharing to **Everyone (public)**.
3. Copy the dashboard URL into the cell below.

In [ ]:
# Paste your dashboard URL here once you've finished building it in AGOL
dashboard_url = 'PASTE_YOUR_DASHBOARD_URL_HERE'

print('Final deliverable URLs:')
print(f'  Feature layer:   {fire_layer_item.homepage}')
print(f'  Cluster summary: {summary_table_item.homepage}')
print(f'  Global metrics:  {metrics_table_item.homepage}')
print(f'  Web map:         {web_map_url}')
print(f'  Dashboard:       {dashboard_url}')
print()
print('Add the dashboard URL to your GitHub README and portfolio.')

## 8. Summary

This notebook turns the static clustering results from Notebook 03 into a published, shareable, interactive deliverable on ArcGIS Online. By the end, the AGOL portal holds:

1. A hosted feature layer of all 1,068 clustered fires with popup-friendly attribute columns (cluster ID, cause, fire size, year, near-shore label).
2. Two CSV tables (cluster summary, global metrics) that the dashboard pulls from for its side panel and headline indicators.
3. A web map styled in the Map Viewer with the project's cluster colors and a configured popup.
4. A dashboard that lets a reviewer click a cluster, see its profile, toggle between basin-wide and near-shore views, and read off the headline finding -- that the arc-length parameter $s$ contributes about 2% on the full basin but jumps an order of magnitude when restricted to fires within 2 km of the lake.

Python's job here was the data prep and the publishing -- the parts where reproducibility matters. The Map Viewer and Dashboards designer handled the styling and layout, where immediate visual feedback beats editing JSON.

The dashboard is the interactive companion to the project; the static figures for the writeup come from Python (matplotlib + geopandas + contextily) and live in the `figures/` directory.